## About this NoteBook

* Contains RoBerta Model for GoEmotions Classification.
* The labels are multiclass and multilabel.
* The dataset is raw and unaugmented.

In [1]:
#Importing the Libraries

!pip install simpletransformers
!pip install transformers-interpret

import pandas
import numpy
from ast import literal_eval
from sklearn.metrics import f1_score, classification_report
from simpletransformers.classification import MultiLabelClassificationModel
from transformers_interpret import SequenceClassificationExplainer

In [2]:
trainPath = "../input/multilabel-data/RawTrain2.csv"
testPath = "../input/multilabel-data/RawTest2.csv"

emoList = ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 
'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism',
'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']

In [3]:
train_df = pandas.read_csv(trainPath)

trainCol = []
for label in train_df['label']:
    tempL = [0] * len(emoList)
    for indiLab in literal_eval(label):
        tempL[emoList.index(indiLab)] = 1
    trainCol.append(tempL)
train_df['label_desc'] = train_df['label']
train_df['label'] = trainCol
train_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

train_df.head()

In [4]:
test_df = pandas.read_csv(testPath)

testCol = []
for label in test_df['label']:
    tempL = [0] * len(emoList)
    for indiLab in literal_eval(label):
        tempL[emoList.index(indiLab)] = 1
    testCol.append(tempL)
test_df['label_desc'] = test_df['label']
test_df['label'] = testCol
test_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

test_df.head()

In [6]:
# Training Roberta Transformer on GoEmotions as MultiLabel Classification

model = MultiLabelClassificationModel('roberta', 'roberta-large', num_labels=len(emoList), 
                                      args={'train_batch_size':4, 'fp16': False, 'overwrite_output_dir': True,                                         
                                            'gradient_accumulation_steps':8, 'learning_rate': 3e-5,
                                            'num_train_epochs': 1, "reprocess_input_data": True, 
                                            'max_seq_length': 128})

model.train_model(train_df)
result, model_outputs, wrong_predictions = model.eval_model(test_df)
predictions, raw_outputs = model.predict(list(test_df['text']))

print('-'*100)

print(f"Accuracy & Loss - {result}")
print(f"F1-Micro Score - {f1_score(test_df['labels'], predictions, average='micro')}")
print(f"F1-Macro Score - {f1_score(test_df['labels'], predictions, average='macro')}")

In [17]:
idx = 16
cls_explainer = SequenceClassificationExplainer(model.model, model.tokenizer, custom_labels=emoList)
word_attributions = cls_explainer(test_df.iloc[idx]['text'])
label = emoList[int(cls_explainer.predicted_class_index)]
word_attributions

In [18]:
print(f"Text : {test_df.iloc[idx]['text']}")
print('-'*100)
print(f"True Label : {test_df.iloc[idx]['label_desc']}")
print(f"Predicted Label : {label}")